In [ ]:
import pandas as pd
import numpy as np
import math
import duckdb
from datetime import datetime, timedelta, timezone

# ==============================================================================
# 1. CORE ASTRONOMICAL ENGINE
# ==============================================================================

def calculate_julian_date(dt):
    """Convert UTC-aware datetime to Julian Date."""
    # Ensure we are working with UTC
    dt = dt.astimezone(timezone.utc)
    year, month, day = dt.year, dt.month, dt.day + dt.hour/24.0 + dt.minute/1440.0 + dt.second/86400.0
    if month <= 2:
        year -= 1
        month += 12
    A = int(year / 100)
    B = 2 - A + int(A / 4)
    return int(365.25 * (year + 4716)) + int(30.6001 * (month + 1)) + day + B - 1524.5

def get_astronomical_state(jd):
    """Calculate geocentric ecliptic longitudes and lunar declination."""
    T = (jd - 2451545.0) / 36525.0
    
    # Sun Longitude
    sun_lon = (280.466 + 36000.77 * T + 1.91 * np.sin(np.radians(357.5 + 35999 * T))) % 360
    
    # Moon Longitude & Declination
    moon_lon = (218.31 + 481267 * T + 6.28 * np.sin(np.radians(134.9 + 477198 * T))) % 360
    moon_lat = 5.12 * np.sin(np.radians(93.2 + 483202 * T))
    epsilon = 23.44 # Obliquity
    sin_dec = np.sin(np.radians(moon_lat)) * np.cos(np.radians(epsilon)) + \
              np.cos(np.radians(moon_lat)) * np.sin(np.radians(epsilon)) * np.sin(np.radians(moon_lon))
    moon_dec = np.degrees(np.arcsin(sin_dec))
    
    # Planetary Longitudes (Mean + Equation of Center)
    def planet_lon(L_mean, pi, e):
        return (L_mean + np.degrees(2 * e * np.sin(np.radians(L_mean - pi)))) % 360

    return {
        'sun_lon': sun_lon,
        'moon_lon': moon_lon,
        'moon_dec': moon_dec,
        'mars_lon': planet_lon(355.4 + 19140 * T, 336.0, 0.093),
        'jup_lon': planet_lon(34.3 + 3034 * T, 14.3, 0.048),
        'sat_lon': planet_lon(50.0 + 1222 * T, 92.0, 0.055)
    }

def calculate_s_score(pos):
    """
    S = 0.13(Syzygy) + 0.10(Dec) - 0.28(Saturn) - 0.15(Jupiter) + 0.04(Mars)
    """
    # Syzygy: Harmonic proximity to 0/180 elongation
    elongation = abs(pos['moon_lon'] - pos['sun_lon'])
    syzygy = (1 + np.cos(np.radians(2 * elongation))) / 2
    
    # S Components
    s = (0.13 * syzygy) + \
        (0.10 * abs(pos['moon_dec'])) - \
        (0.28 * np.sin(np.radians(pos['sat_lon']))) - \
        (0.15 * np.sin(np.radians(pos['jup_lon']))) + \
        (0.04 * np.sin(np.radians(pos['mars_lon'])))
    return s

# ==============================================================================
# 2. DATA GENERATION & UNIFICATION
# ==============================================================================

# Time Range: 1900 to 2025
start_dt = datetime(1900, 1, 1, tzinfo=timezone.utc)
end_dt = datetime(2025, 12, 31, tzinfo=timezone.utc)
total_seconds = int((end_dt - start_dt).total_seconds())

# A. SYNTHETIC STREAM (50,000 UNIQUE TIMESTAMPS)
print("Generating 50,000 Synthetic records (Background Noise)...")
rng = np.random.default_rng(42)
random_seconds = rng.choice(total_seconds, size=50000, replace=False)
synthetic_data = []

for i, sec in enumerate(random_seconds):
    dt = start_dt + timedelta(seconds=int(sec))
    state = get_astronomical_state(calculate_julian_date(dt))
    synthetic_data.append({
        'timestamp': dt,
        'record_type': 'Synthetic',
        'place': 'Monte Carlo High-Res Sample',
        'magnitude': None,
        's_score': calculate_s_score(state)
    })
    if (i + 1) % 10000 == 0:
        print(f"...{i+1} records generated")

df_synthetic = pd.DataFrame(synthetic_data)

# B. REAL STREAM (ALL EARTHQUAKE RECORDS)
print("Processing RealStream records...")
df_real_in = pd.read_csv('/workspaces/AI-Catastrophe-Analytics/Mega_Quake_Topocentric_Analysis.csv')
# Mixed timestamp formats in the source; parse per-row to avoid format errors.
df_real_in['time'] = pd.to_datetime(df_real_in['time'], utc=True, format='mixed', errors='coerce')

real_data = []
for _, row in df_real_in.dropna(subset=['time']).iterrows():
    state = get_astronomical_state(calculate_julian_date(row['time']))
    real_data.append({
        'timestamp': row['time'],
        'record_type': 'RealStream',
        'place': row['place'],
        'magnitude': row['mag'],
        's_score': calculate_s_score(state)
    })

df_real = pd.DataFrame(real_data)

# C. UNIFIED DATAFRAME
df_unified = pd.concat([df_synthetic, df_real], ignore_index=True)
df_unified['timestamp'] = pd.to_datetime(df_unified['timestamp'], utc=True, errors='coerce')
df_unified['timestamp'] = df_unified['timestamp'].dt.tz_localize(None)

# ==============================================================================
# 3. DUCKDB PERSISTENCE
# ==============================================================================
print("Persisting to DuckDB...")
con = duckdb.connect('mega_quake_unified.db')

con.execute("""
    CREATE OR REPLACE TABLE earthquake_risk_stream AS 
    SELECT
        * EXCLUDE (timestamp),
        CAST(timestamp AS TIMESTAMP) AS timestamp
    FROM df_unified
""")

# Validation Check
results = con.execute("""
    SELECT 
        record_type, 
        COUNT(*) as record_count,
        quantile_cont(s_score, 0.95) as p95,
        quantile_cont(s_score, 0.99) as p99
    FROM earthquake_risk_stream 
    GROUP BY record_type
""").df()

print("\n--- Final Stream Statistics ---")
print(results)

con.close()
print("\nSuccess! Database 'mega_quake_unified.db' is ready.")

Generating 50,000 Synthetic records (Background Noise)...
...10000 records generated
...20000 records generated
...30000 records generated
...40000 records generated
...50000 records generated
Processing RealStream records...


ValueError: time data "1965-02-04 05:01:22+00:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f%z". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [28]:
db_conn = duckdb.connect('mega_quake_unified.db')
res_df = db_conn.execute("""
   SELECT COUNT(1) FROM earthquake_risk_stream 
    WHERE record_type IN  ('Synthetic')
    AND s_score  >= 1.5
                        
""").df()
dis_df = db_conn.execute("""
   SELECT 
    record_type, 
    COUNT(*) AS sample_size,
    quantile_cont(s_score, 0.95) AS p95_threshold, 
    quantile_cont(s_score, 0.99) AS p99_threshold,
    AVG(s_score) AS mean_s,
    MAX(s_score) AS max_s
FROM earthquake_risk_stream
GROUP BY record_type;
                        
""").df()
print(dis_df.head())

  record_type  sample_size  p95_threshold  p99_threshold    mean_s     max_s
0  RealStream           83       2.778143       2.979405  1.539525  2.995777
1   Synthetic       100000       2.813572       3.090008  1.576211  3.426976


In [30]:
dis_df = db_conn.execute("""
   WITH synthetic_threshold AS (
    SELECT quantile_cont(s_score, 0.95) as p95_val
    FROM earthquake_risk_stream
    WHERE record_type = 'Synthetic'
)
SELECT 
    timestamp, 
    place, 
    magnitude, 
    s_score,
    (s_score - (SELECT p95_val FROM synthetic_threshold)) as anomaly_strength
FROM earthquake_risk_stream
WHERE record_type = 'RealStream' 
  AND s_score > (SELECT p95_val FROM synthetic_threshold)
ORDER BY s_score DESC;
                        
""").df()
print(dis_df.head())
con.close()

  timestamp                                      place  magnitude   s_score  \
0       NaT            1989 Macquarie Ridge Earthquake       8.02  2.995777   
1       NaT  121 km SSE of San Pedro de Atacama, Chile       8.20  2.975811   
2       NaT             25 km WSW of Valparaíso, Chile       8.00  2.915138   
3       NaT            1932 Jalisco, Mexico Earthquake       8.10  2.837280   

   anomaly_strength  
0          0.182205  
1          0.162239  
2          0.101567  
3          0.023708  
